# Reproducing the IBL brain-wide map (2025) from dandiset 000409

**Paper.** International Brain Laboratory, Angelaki D, Benson B, et al. (2025). A brain-wide map of
neural activity during complex behaviour. *Nature* 645, 177-191.
[doi:10.1038/s41586-025-09235-0](https://doi.org/10.1038/s41586-025-09235-0). This attempt used the
full text of the version of record, which is open access (CC BY 4.0).

**Dandiset.** [DANDI:000409](https://dandiarchive.org/dandiset/000409/0.260309.1324), version
0.260309.1324. The paper does not cite a DANDI version (its data statement points to the IBL's own
data release). 0.260309.1324 is the only published version. The draft holds the same 2,048 files
with the same paths and sizes; its metadata differ (the draft links the Nature paper, the published
version the bioRxiv preprint), none of which the analysis uses.

**What the dandiset holds.** For each of 459 sessions a processed file (trials, spike-sorted units of
all probes with their Allen CCF localisation, wheel, licks, and for most sessions video pose, pupil
and motion energy), a raw electrophysiology file, and up to three videos. This notebook streams the
processed files only.

**Outside inputs.** The paper's table of per-region results, `brainwidemap/meta/region_info.csv` in
https://github.com/int-brain-lab/paper-brain-wide-map at commit 118fc36, and its table of inclusion
counts `meta/inclusion_info.csv`, and its trajectory results table `meta/Results/stim.pkl`, read only as the
paper's reported values. No measurement comes from
outside the dandiset.

**Scope.** The neural analyses of the paper run over 454 sessions and 201 regions with 10 repeats of
nested cross-validation and 200 null sessions each; that does not fit the 30-minute limit. The
decoding, single-cell and trajectory claims are tested on a set of regions fixed in the claims file,
each with all its sessions. The stimulus trajectories are computed for every canonical region. Four claims that need every region
and heavier models were not attempted (blocked, `other`); the cost measured here is printed at the end.

| Claim | Tested |
|---|---|
| dataset-sessions-mice-labs: 459 sessions, 139 mice, 12 labs | yes |
| dataset-probes-units: 699 insertions, 621,733 units | yes |
| well-isolated-neurons: 75,708 well-isolated neurons | yes |
| canonical-regions: 201 regions, 62,857 neurons | yes |
| brain-areas-279: 279 brain areas | yes |
| mice-sex: 94 male, 45 female | yes |
| mice-age-weight: 13 to 178 weeks, 16.1 to 35.7 g | yes |
| probe-versions: 109 3A, 586 3B2, 4 2.4 | yes |
| trials-per-session: mean 645, median 602, 401 to 1,525 | yes |
| rf-mapping: receptive fields (Supplementary Fig. 5) | blocked, data missing |
| behav-performance: 81.4% correct | yes |
| behav-zero-contrast: 58.7% at 0% contrast, t138 = 20.18 | yes |
| behav-early-movements: 22.8% of first movements under 80 ms | yes |
| fig1d-contrast: better and faster at high contrast | yes |
| fig1ef-end-of-session: worse and slower at session end | yes |
| fig1g-block-switch: 5 to 10 trials to adjust | yes |
| fig1h-block-bias: block biases choices at all contrasts | yes |
| dec-stim-restricted: stimulus decodable in fewer regions, VISp | yes, 10 regions |
| dec-choice-widespread: choice in more regions than stimulus, GRN | yes, 10 regions |
| dec-feedback-nearly-all: feedback in nearly all regions | yes, 10 regions |
| sc-stim-fraction: few stimulus-selective neurons | yes, 14 regions |
| sc-choice-fraction: more choice- than stimulus-selective neurons | yes, 14 regions |
| traj-latency-visual: LGd < VISp = LP < VISpm < VISam | yes |
| traj-latency-late: IRN, GRN at 100 to 120 ms | yes, 2 of 5 regions |
| traj-stim-regions: 104 regions with stimulus trajectories | yes, all canonical regions |
| fback-oscillation-licking: about 10 Hz with licking | yes, IRN |
| wheel-speed-decoding: 163 of 201 regions | blocked, not attempted |
| encoding-feedback-largest: correct-feedback kernel largest | blocked, not attempted |
| video-correlates: half of neurons correlate with video | blocked, not attempted |
| granger-interactions | blocked, not attempted |

In [ ]:
import os
os.environ.update(OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1")
import io
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from scipy import optimize, stats

import bwm

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)

REPO = "https://raw.githubusercontent.com/int-brain-lab/paper-brain-wide-map/118fc36cb3602934466ad2c6087c2b3b441f9f1f"
region_info = pd.read_csv(io.StringIO(requests.get(f"{REPO}/brainwidemap/meta/region_info.csv", timeout=60).text))
inclusion_info = pd.read_csv(io.StringIO(requests.get(f"{REPO}/brainwidemap/meta/inclusion_info.csv", timeout=60).text), index_col=0)

started = time.time()
assets = bwm.list_assets()
processed = assets[assets.kind == "processed"].sort_values("path").reset_index(drop=True)
scans = bwm.run_parallel(bwm.scan_session, list(processed.asset_id))
scan_seconds = time.time() - started
print(f"dandiset {bwm.DANDISET_ID} version {bwm.DANDISET_VERSION}: {len(assets)} assets; "
      f"scanned {len(scans)} processed files in {scan_seconds:.0f} s")

In [ ]:
# One row per session, and the trials of all sessions in the paper's terms.
rows, trial_tables = [], []
for s in scans:
    i, u = s["info"], s["units"]
    t = bwm.task_table(s["trials"])
    mask, n_kept = bwm.analysis_mask(t)
    t["analysed"] = mask
    t["eid"] = i["session_id"]
    t["subject"] = i["subject_id"]
    t["trial"] = np.arange(len(t))
    t["from_end"] = len(t) - t.trial
    trial_tables.append(t)
    rows.append({"eid": i["session_id"], "subject": i["subject_id"], "lab": i["lab"], "sex": i["sex"],
                 "dob": pd.Timestamp(i["date_of_birth"]), "start": pd.Timestamp(i["session_start"]),
                 "weight_g": float(i["weight"].split()[0]) * 1000 if i["weight"] else np.nan,
                 "protocol": i["protocol"], "revision": i["revision"], "n_trials": len(t),
                 "n_kept": n_kept, "n_analysed": int(mask.sum()), "n_units": len(u),
                 "n_good": int((u.ibl_quality_score == 1).sum()), "n_probes": u.probe_name.nunique(),
                 "modules": ",".join(i["processing"]),
                 "passive": ",".join(s["passive"]) if s["passive"] else "none"})
sessions = pd.DataFrame(rows)
trials = pd.concat(trial_tables, ignore_index=True)
sessions["start"] = pd.to_datetime(sessions.start, utc=True)
sessions["dob"] = pd.to_datetime(sessions.dob, utc=True)
sessions["age_weeks"] = (sessions.start - sessions.dob).dt.days / 7
print(sessions.drop(columns=["modules", "passive"]).head().to_string())

## Data check

Each check of the metadata against the data is stated under its output.

In [ ]:
print("Assets by kind:", assets.kind.value_counts().to_dict())
per_session = assets.groupby("eid").kind.apply(lambda k: ",".join(sorted(k)))
print("Sessions with a processed and a raw file:", int((per_session.str.contains("processed") & per_session.str.contains("raw")).sum()),
      "of", per_session.size)
print("Sessions:", len(sessions), "; subjects:", sessions.subject.nunique(), "; labs:", sessions.lab.nunique())
print("Sessions per lab:", sessions.lab.value_counts().to_dict())
print("Sessions per subject:", sessions.subject.value_counts().value_counts().sort_index().to_dict(), "(sessions: subjects)")
print("Spike-sorting revision in the files:", sessions.revision.value_counts().to_dict())
print("Task protocols:", sessions.protocol.str.replace("_iblrig_tasks_", "").value_counts().to_dict())
print("Processing modules:", sessions.modules.value_counts().to_dict())
print("Passive protocol phases:", sessions.passive.value_counts().to_dict())
print("Groups under stimulus/ (presentation, templates) in the processed files:", pd.Series([s["stimulus"] for s in scans]).value_counts().to_dict())
epochs = pd.concat([s["epochs"] for s in scans])
print("Epoch types:", epochs.protocol_type.value_counts().to_dict())

The counts of sessions, mice and labs are those of the paper. The spike sorting in the files is IBL
revision 2025-05-06; the paper's code loads revision 2024-05-06. The unit counts below match the
paper, so this does not change which units are well-isolated, as far as counts show. No session
lists receptive-field mapping among its passive phases (only spontaneous activity and task replay),
and the processed files hold no receptive-field stimulus, so the RF analysis cannot be repeated.
Pose, pupil and motion energy are missing from some sessions (see the module list).

In [ ]:
subj = sessions.groupby("subject").agg(sex=("sex", "first"), n_sex=("sex", "nunique"), dob=("dob", "nunique"),
                                       w_min=("weight_g", "min"), w_max=("weight_g", "max"), n=("eid", "size"))
print("Sex per mouse:", subj.sex.value_counts().to_dict(), "; mice with more than one sex or birth date:",
      int((subj.n_sex > 1).sum()), int((subj.dob > 1).sum()))
print("Weight in the files (g): range", sessions.weight_g.min(), "to", sessions.weight_g.max(),
      "; sessions without weight:", int(sessions.weight_g.isna().sum()))
print("Mice whose weight differs between their sessions:", int((subj.w_max > subj.w_min).sum()), "of", int((subj.n > 1).sum()),
      "with more than one session")
print("Age at recording (weeks): range", round(sessions.age_weeks.min(), 1), "to", round(sessions.age_weeks.max(), 1))

Every mouse has one sex and one date of birth. The weight field holds one value per mouse, the same
in all its sessions, so it is not the weight on each recording day.

In [ ]:
print("Trial columns:", list(scans[0]["trials"].columns))
print("Trials:", len(trials), "; per session", sessions.n_trials.min(), "to", sessions.n_trials.max())
print("Contrasts (%):", (trials.contrast * 100).value_counts().sort_index().to_dict())
c = trials[trials.p_left != 0.5].contrast.value_counts()
print("Ratio of contrasts 1, 0.25, 0.125, 0.0625, 0 in biased blocks:", (c.loc[[1, 0.25, 0.125, 0.0625, 0]] / c.loc[0]).round(2).tolist())
print("p_left values:", trials.p_left.value_counts().to_dict())
first = trials.groupby("eid").apply(lambda g: int((g.block == 0).sum()), include_groups=False)
print("Length of the first (unbiased) block:", first.value_counts().to_dict())
biased = trials[trials.p_left != 0.5]
match = biased.groupby("eid").apply(lambda g: np.mean((g.side < 0) == (g.p_left == 0.8)), include_groups=False)
sessions["block_match"] = sessions.eid.map(match)
sessions["block_ok"] = sessions.block_match >= 0.7
BLOCK_OK = set(sessions.eid[sessions.block_ok])
print("Fraction of biased-block trials whose stimulus is on the block's likely side, per session:")
print("   sessions at or above 0.7:", len(BLOCK_OK), "range", round(match[match >= 0.7].min(), 3), "to", round(match[match >= 0.7].max(), 3))
print("   sessions below 0.7:", int((~sessions.block_ok).sum()), "range", round(match[match < 0.7].min(), 3), "to", round(match[match < 0.7].max(), 3))
bad_s = sessions[~sessions.block_ok]
print("   their protocols:", bad_s.protocol.value_counts().to_dict(), "; labs:", bad_s.lab.value_counts().to_dict(),
      "; dates", bad_s.start.min().date(), "to", bad_s.start.max().date())
print("   all sessions with that protocol name:", int((sessions.protocol == "_iblrig_tasks_ephysChoiceWorld").sum()))
blen = trials[trials.block > 0].groupby(["eid", "block"]).size()
last = trials.groupby("eid").block.max()
for name, ok in (("block_ok", True), ("other", False)):
    b = blen[[(e in BLOCK_OK) == ok and bl < last[e] for e, bl in blen.index]]
    print(f"Biased block lengths in {name} sessions, last block left out: min {b.min()}, max {b.max()}, mean {b.mean():.1f}, blocks {len(b)}")
print("Stimulus side x choice x rewarded:")
print(pd.crosstab([trials.side, trials.choice], trials.correct).to_string())
print("Reward volume on rewarded trials (uL):", trials[trials.correct].reward_ul.round(2).value_counts().head(6).to_dict())
q = pd.concat([s["trials"].quiescence_period for s in scans])
print("Quiescence period (s): range", round(q.min(), 3), "to", round(q.max(), 3))
print("Missing values:", {k: int(trials[k].isna().sum()) for k in ["stim_on", "go_cue", "first_move", "feedback_time"]})
print("Trials without a response:", int((trials.choice == 0).sum()))
print("Stimulus onset minus go cue (ms), median and 1st/99th percentiles:",
      np.nanpercentile((trials.stim_on - trials.go_cue) * 1000, [50, 1, 99]).round(1).tolist())
print("First movement before stimulus onset:", round(float(np.mean(trials.rt < 0)) * 100, 2), "% of trials; before the go cue:",
      round(float(np.mean(trials.first_move < trials.go_cue)) * 100, 2), "%")
print("Trials analysed (complete events, first movement 0.08 to 2 s):", int(trials.analysed.sum()), "of", len(trials))
trunc = sessions[sessions.n_kept < sessions.n_trials]
print("Sessions truncated to pass the 90% criterion on easy (contrast >= 0.5) trials:", len(trunc))
for r in trunc.itertuples():
    tt = trials[trials.eid == r.eid].reset_index(drop=True)
    print(f"   {r.eid[:8]}: {r.n_trials} trials, kept {r.n_kept} (floor 400: {r.n_kept == 400}); easy-trial performance "
          f"{bwm.perf_easy(tt):.3f} over all trials, {bwm.perf_easy(tt.iloc[:r.n_kept]):.3f} over those kept; analysed {r.n_analysed}")
print("Sessions that do not pass even at 400 trials, and are excluded as the code excludes them:", int((sessions.n_analysed == 0).sum()))

In 64 sessions, every one recorded with the protocol named `_iblrig_tasks_ephysChoiceWorld`
without a version, `probability_left` alternates between 0.2 and 0.8 on every trial after the first
90, and the stimulus falls on the stated block's likely side on about half of the trials, as by
chance; in the other 395 sessions it does so on 80% of trials, with blocks of 20 to 100 trials as
the Methods state. The block column of those 64 sessions does not describe the blocks the mice
experienced. Analyses that need the block (Figure 1g and 1h, the single-cell tests and the
trajectory permutations) leave these sessions out; the decoding nulls are drawn from the generative
process and do not use the recorded blocks.

The stimulus, choice and reward columns agree: a left stimulus is rewarded after a clockwise turn
and a right stimulus after a counter-clockwise turn, as the table's description says (clockwise moves
the stimulus right, towards the centre). The choice called right here is the counter-clockwise turn.
The contrasts and block probabilities are those of the Methods, and the first block is the 90
unbiased trials. Two descriptions disagree with the data. The first movement time is described as
"after the go cue", but in a part of the trials it precedes both the go cue and the stimulus onset
(printed above); the paper's 80 ms threshold removes these. The reward is 3 uL in the Methods, while
the files record the volumes printed above. Neither is used by an analysis that depends on it,
other than through the paper's own trial mask. Two sessions are truncated to pass the 90% criterion
on easy trials, to 457 and 460 trials, both above the 400-trial floor; no session fails the criterion
at 400 trials, so none is excluded for it.

In [ ]:
units = pd.concat([s["units"].assign(eid=s["info"]["session_id"]) for s in scans], ignore_index=True)
passes = ((units.median_spike_amplitude_uV > 50).astype(int) + (units.noise_cutoff < 20).astype(int)
          + (units.sliding_rp_violation == 1).astype(int)) / 3
print("Units:", len(units), "; quality score values:", units.ibl_quality_score.round(2).value_counts().sort_index().to_dict())
print("Units whose score equals the fraction of the three metrics they pass, noise cutoff below 20 (Methods):",
      int(np.isclose(passes, units.ibl_quality_score, atol=0.01).sum()), "of", len(units))
passes5 = ((units.median_spike_amplitude_uV > 50).astype(int) + (units.noise_cutoff < 5).astype(int)
           + (units.sliding_rp_violation == 1).astype(int)) / 3
print("The same with a noise cutoff below 5:", int(np.isclose(passes5, units.ibl_quality_score, atol=0.01).sum()))
print("Noise cutoff: units with a value", int(units.noise_cutoff.notna().sum()), "; range", round(units.noise_cutoff.min(), 2), "to",
      round(units.noise_cutoff.max(), 1), "; well-isolated units with a cutoff of 5 or more:",
      int(((units.ibl_quality_score == 1) & (units.noise_cutoff >= 5)).sum()))
print("Units whose spike_count equals the length of their spike train:", int((units.spike_count == units.n_spikes_index).sum()))
print("Units whose probe matches the probe of their peak channel:", int((units.probe_name == units.region_probe).sum()))
probe_models = pd.Series([v.split(" probe.")[0].replace("A ", "", 1) for s in scans for v in s["probes"].values()])
print("Probe devices by model:", probe_models.value_counts().to_dict())
print("Units per probe: mean", round(len(units) / sessions.n_probes.sum(), 1))
descs = pd.Series([s["info"]["units_description"] for s in scans])
print("Distinct units-table descriptions:", descs.nunique(), "; first sentence:", descs.iloc[0].split(". ")[0])
print("Descriptions that mention a 'label' column:", int(descs.str.contains("'label' column").sum()), "; columns of the units table include 'label':",
      "label" in scans[0]["units"].columns)

The quality score is the fraction of three metrics passed (median amplitude above 50 uV, the noise
cutoff, the sliding refractory-period test) for every unit when the noise cutoff threshold is 5,
not the 20 uV the Methods give; the column's description gives it in standard deviations, not uV.
The units table's description calls the score a `label` column, which is not in the files, and
describes all units as from Neuropixels 1.0 probes, though four insertions used 2.x probes.

In [ ]:
# Recordings deposited twice: identical trial timings or identical unit spike counts.
fp_trials = trials.groupby("eid").apply(lambda g: (len(g), round(float(g.stim_on.iloc[0]), 4), round(float(g.feedback_time.iloc[-1]), 4)), include_groups=False)
dup_t = fp_trials[fp_trials.duplicated(keep=False)]
fp_units = units.groupby(["eid", "probe_name"]).spike_count.apply(lambda c: tuple(np.sort(c.values)[-20:]))
dup_u = fp_units[fp_units.duplicated(keep=False)]
print("Sessions sharing trial count, first stimulus time and last feedback time with another:", len(dup_t))
print("Probe insertions sharing their 20 largest unit spike counts with another:", len(dup_u))
same_day = sessions.groupby(["subject", sessions.start.dt.date]).size()
print("Mice with more than one session on one day:", int((same_day > 1).sum()))

No session or insertion repeats another, so each recording is counted once. Units are taken per
session; no analysis here treats a unit as the same across sessions.

In [ ]:
# Clocks of the wheel and lick streams against the trials table.
near, lick_ratio = [], []
for s in scans:
    t = bwm.task_table(s["trials"])
    fm = t.first_move.dropna().values
    wm = np.sort(s["wheel_moves"])
    if len(wm) and len(fm):
        j = np.clip(np.searchsorted(wm, fm), 1, len(wm) - 1)
        near.append(np.minimum(np.abs(wm[j] - fm), np.abs(wm[j - 1] - fm)))
    lk = np.sort(s["licks"])
    fb = t.feedback_time[t.correct].dropna().values
    if len(lk) > 100:
        after = (np.searchsorted(lk, fb + 1) - np.searchsorted(lk, fb)).sum()
        before = (np.searchsorted(lk, fb) - np.searchsorted(lk, fb - 1)).sum()
        lick_ratio.append(after / max(before, 1))
near = np.concatenate(near)
print("First movement time to the nearest wheel-movement onset (ms): median", round(np.median(near) * 1000, 2),
      "; within 50 ms:", round(np.mean(near < 0.05) * 100, 1), "% of", len(near), "trials")
print("Licks in the 1 s after reward over the 1 s before, per session: median", round(np.median(lick_ratio), 1),
      "; sessions above 2:", int(np.sum(np.array(lick_ratio) > 2)), "of", len(lick_ratio))

The trials' first movement times coincide with onsets in the wheel's movement intervals (96.8%
within 50 ms), and licks rise after reward in 407 of 442 sessions with licks, so the wheel and lick
streams share the trials' clock. The spikes are checked against the trials' clock after they are loaded, below.

## dataset-sessions-mice-labs

> The data come from 459 sessions of 139 mice recorded in 12 laboratories.

In [ ]:
print("Sessions:", len(sessions), "(paper 459); mice:", sessions.subject.nunique(), "(139); labs:", sessions.lab.nunique(), "(12)")

**Result: reproduced.** The dandiset holds 459 sessions of 139 mice from 12 labs, as the paper states.

## dataset-probes-units

> 699 Neuropixels insertions yielded 621,733 units, an average of 889 per probe.

In [ ]:
n_probes = int(sessions.n_probes.sum())
print("Insertions:", n_probes, "(paper 699); units:", len(units), "(621,733); per probe:", round(len(units) / n_probes, 1), "(889)")

**Result: reproduced.** 699 insertions and 621,733 units, 889.5 per probe.

## well-isolated-neurons

> Stringent quality metrics identified 75,708 well-isolated neurons, an average of 108 per probe.

In [ ]:
n_good = int((units.ibl_quality_score == 1).sum())
print("Well-isolated units:", n_good, "(paper 75,708); per probe:", round(n_good / n_probes, 1), "(108)")
print("Well-isolated by recomputing the three metrics: noise cutoff below 20 (Methods)", int((passes == 1).sum()),
      "; below 5", int((passes5 == 1).sum()))

**Result: reproduced.** 75,708 units have a quality score of 1, 108.3 per probe.

## canonical-regions

> The main analyses use a canonical dataset of 201 regions with at least 2 sessions of 5
> well-isolated neurons and at least 20 neurons in total, 62,857 neurons.

In [ ]:
canon, steps = bwm.canonical_units(scans)
paper_steps = inclusion_info.iloc[[0, 3, 4, 5, 6, 7]].reset_index(drop=True)
steps["paper_sessions"] = paper_steps.n_sessions.values
steps["paper_units"] = paper_steps.n_units.values
steps["paper_regions"] = paper_steps.n_regions.values
print(steps.to_string(index=False))
canon_regions = sorted(canon.region.unique())
paper_canonical = set(region_info[region_info.canonical].Beryl)
print("\nCanonical regions:", len(canon_regions), "; regions marked canonical in region_info.csv:", len(paper_canonical),
      "; of ours among them:", len(set(canon_regions) & paper_canonical))
print("Marked canonical in region_info.csv but with fewer than 20 units:", sorted(paper_canonical - set(canon_regions)))

**Result: reproduced.** Every inclusion step gives the counts of the paper's inclusion table, ending
with 201 regions and 62,857 neurons. The `canonical` column of the paper's region table marks 210
regions, those left before the 20-neuron step.

## brain-areas-279

> The probes covered 279 brain areas.

In [ ]:
grey = units[~units.region.isin(["void", "root"])]
print("Beryl regions with at least one unit, void and root left out:", grey.region.nunique(), "(paper 279)")
print("With at least one well-isolated unit:", grey[grey.ibl_quality_score == 1].region.nunique())

**Result: reproduced.** Units fall in 279 Beryl regions other than void and root.

## mice-sex

> The 139 mice were 94 male and 45 female.

In [ ]:
print(subj.sex.value_counts().to_dict(), "(paper 94 male, 45 female)")

**Result: reproduced.** 94 male and 45 female mice.

## mice-age-weight

> On the day of recording the mice were aged 13 to 178 weeks (mean 44.96, median 27.0) and weighed
> 16.1 to 35.7 g (mean 23.9, median 23.84).

In [ ]:
s_sorted = sessions.sort_values("start")
readings = {"a: first session": s_sorted.groupby("subject").first(), "b: last session": s_sorted.groupby("subject").last(),
            "c: every session": s_sorted}
expected = {"age_weeks": [13, 178, 44.96, 27.0], "weight_g": [16.1, 35.7, 23.9, 23.84]}
for name, d in readings.items():
    for var, exp in expected.items():
        v = d[var].dropna()
        got = [v.min(), v.max(), v.mean(), v.median()]
        ok = [abs(g - e) <= 0.1 * e for g, e in zip(got, exp)]
        print(f"{name:17s} {var:9s} min {got[0]:6.1f} max {got[1]:6.1f} mean {got[2]:6.2f} median {got[3]:6.2f}  "
              f"within 10%: {ok}  (n = {len(v)})")
print("Paper's code (meta/mice_info.py) takes age_weeks and reference_weight from the IBL database, not from the recording day.")

**Result: partially reproduced.** Weight holds under every reading: per mouse at the first session
16.1 to 36.2 g, mean 24.19 and median 24.00, all within 10% of the paper's values. The files give
each mouse one weight for all its sessions, so this is not a weight measured on each recording day.
Age does not hold: at the first session the mice were 13.1 to 63.1 weeks old (mean 28.09, median
26.00), and no reading comes near the paper's maximum of 178 weeks or mean of 44.96 (at most 65.9
and 28.86). The paper's code takes age from the IBL database, not from the recording day, which may
explain the difference.

## probe-versions

> The insertions used Neuropixels 1.0 probes of versions 3A (109) and 3B2 (586), and Neuropixels
> 2.4 probes (4).

In [ ]:
group = probe_models.map(lambda m: "3A (NP1010)" if m == "NP1010" else "3B2 (PRB_1_4_0480_1)" if m.startswith("PRB_1_4_0480_1")
                         else "2.x (NP2010)" if m == "NP2010" else m)
print(group.value_counts().to_dict(), "(paper 3A 109, 3B2 586, 2.4 4)")

**Result: reproduced.** The model groups hold 586, 109 and 4 insertions.

## trials-per-session

> Recorded sessions lasted on average 645 trials (median 602, range 401 to 1,525).

In [ ]:
n = sessions.n_trials
print(f"mean {n.mean():.1f} (645), median {n.median():.0f} (602), range {n.min()} to {n.max()} (401 to 1,525)")

**Result: reproduced.** Mean 645.1, median 601, range 401 to 1,525 trials.

## rf-mapping (blocked)

> Receptive-field mapping after the task found a relatively small fraction of cells with significant
> receptive fields.

Blocked, data missing: no session holds the receptive-field mapping phase or its stimulus (see the
data check). The paper used it in 504 of 699 insertions.

## behav-performance

> After training, mice made correct choices on 81.4 +/- 0.4% (mean +/- s.d.) of trials.

In [ ]:
def summary(v):
    v = np.asarray(v, float)
    return f"mean {v.mean():.2f}%, s.d. {v.std(ddof=1):.2f}, s.e.m. {v.std(ddof=1) / np.sqrt(len(v)):.2f} (n = {len(v)})"

perf = {
    "a: mice, all trials": trials.groupby("subject").correct.mean() * 100,
    "b: mice, analysed trials": trials[trials.analysed].groupby("subject").correct.mean() * 100,
    "c: sessions, all trials": trials.groupby("eid").correct.mean() * 100,
}
for k, v in perf.items():
    print(f"{k:26s}", summary(v), " within 81.4 +/- 0.4:", abs(v.mean() - 81.4) <= 0.4)
pooled = trials.correct.mean() * 100
print(f"{'d: all trials pooled':26s} {pooled:.2f}%  within 81.4 +/- 0.4: {abs(pooled - 81.4) <= 0.4}")
canon_t = trials[trials.eid.isin(set(canon.eid))]
v454 = {"a: mice, all trials": canon_t.groupby("subject").correct.mean() * 100,
        "c: sessions, all trials": canon_t.groupby("eid").correct.mean() * 100}
print(f"Variant on the {canon_t.eid.nunique()} sessions of the canonical set ({canon_t.subject.nunique()} mice):")
for k, v in v454.items():
    print(f"   {k:26s}", summary(v), " within 81.4 +/- 0.4:", abs(v.mean() - 81.4) <= 0.4)

**Result: reproduced.** Under reading (c), the mean over sessions, 81.58% of trials are correct,
within 81.4 +/- 0.4. The mean over mice is 80.76% over all trials and 84.62% over analysed trials,
both outside the tolerance, and all trials pooled give 81.94%. Across mice the s.d. is 5.12 and the
s.e.m. 0.43, so the paper's +/- 0.4 matches an s.e.m., not the s.d. it states. On the 454 sessions of
the canonical set the readings barely move (mean over mice 80.77%, over sessions 81.59%).

## behav-zero-contrast

> On 0% contrast trials mice were rewarded on 58.7 +/- 0.4% (mean +/- s.e.m. across mice) of trials,
> better than chance (t138 = 20.18, P = 5.2e-43).

In [ ]:
canon_t = trials[trials.eid.isin(set(canon.eid))]
for name, d in (("a: all trials", trials), ("b: analysed trials", trials[trials.analysed]),
                ("variant, a on the 454 canonical sessions", canon_t), ("variant, b on the 454", canon_t[canon_t.analysed])):
    v = d[d.contrast == 0].groupby("subject").correct.mean() * 100
    t_, p_ = stats.ttest_1samp(v, 50)
    print(f"{name:41s} {summary(v)}; t{len(v) - 1} = {t_:.2f}, P = {p_:.2g}; "
          f"mean within 0.4: {abs(v.mean() - 58.7) <= 0.4}, t within 2.02: {abs(t_ - 20.18) <= 2.02}")
print("Paper: t138 = 20.18 gives P =", f"{stats.t.sf(20.18, 138) * 2:.2g}", "two-sided")

**Result: partially reproduced.** Mice are above chance at 0% contrast. Over all trials the mean over
139 mice is 58.26% (s.e.m. 0.42), t138 = 19.68, P = 6.9e-42, and the t statistic is within its
tolerance; the mean is 0.44 points below the paper's 58.7, just outside the 0.4 tolerance. Over
analysed trials the mean is 59.29% and t138 = 16.21. Neither reading puts both values within
tolerance. On the 454 sessions of the canonical set, which the Fig. 1d legend names, the values are
58.27% and t138 = 19.68 (all trials) and 59.31% and 16.23 (analysed trials), so the session set does
not explain the difference.

## behav-early-movements

> 22.8% of first wheel-movement times occurred under 80 ms after stimulus onset.

In [ ]:
early = trials.rt < 0.08
has = trials.rt.notna()
print(f"a: over all trials {early.mean() * 100:.2f}% of {len(trials)}")
print(f"b: over trials with both times {early[has].mean() * 100:.2f}% of {int(has.sum())}")
print(f"c: mean over sessions {trials[has].groupby('eid').rt.apply(lambda r: np.mean(r < 0.08)).mean() * 100:.2f}%")
print(f"Of which before stimulus onset: {np.mean(trials.rt[has] < 0) * 100:.2f}% of trials with both times")

**Result: reproduced.** 22.75% of all 296,090 trials have a first movement time under 80 ms
(reading a), against the paper's 22.8%; reading (b) gives 22.99% and (c) 22.49%. 12.06% of trials
with both times have the first movement before the stimulus onset.

## fig1d-contrast

> Mice performed better and faster on trials with high visual contrast.

In [ ]:
resp = trials[trials.choice != 0]
by_c = resp.groupby(["subject", "contrast"]).agg(correct=("correct", "mean"), rt=("rt", "median")).reset_index()
pc = by_c.pivot(index="subject", columns="contrast", values="correct") * 100
rt = by_c.pivot(index="subject", columns="contrast", values="rt") * 1000
print("Median over mice, percent correct by contrast:", pc.median().round(1).to_dict())
print("Median over mice, median first movement time (ms) by contrast:", rt.median().round(0).to_dict())
w1 = stats.wilcoxon(pc[1.0], pc[0.0])
w2 = stats.wilcoxon(rt[1.0], rt[0.0])
print(f"100% against 0%: correct higher in {(pc[1.0] > pc[0.0]).sum()} of {len(pc)} mice, p = {w1.pvalue:.2g}; "
      f"first movement shorter in {(rt[1.0] < rt[0.0]).sum()} of {len(rt)} mice, p = {w2.pvalue:.2g}")
print("Median percent correct rises at every contrast step:", bool(np.all(np.diff(pc.median().values) > 0)))
print("Median first movement time falls at every contrast step:", bool(np.all(np.diff(rt.median().values) < 0)))

**Result: reproduced.** The median over mice of percent correct rises at every contrast step, from
59.2% at 0% to 96.8% at 100% (higher at 100% in 139 of 139 mice, p = 1.5e-24), and the median first
movement time falls at every step, from 196 to 137 ms (shorter at 100% in 133 of 139 mice,
p = 5.9e-24).

## fig1ef-end-of-session

> Towards the end of the sessions, performance decreased and first wheel-movement times increased.

In [ ]:
nz = trials[(trials.contrast > 0) & (trials.choice != 0)]
end = nz[nz.from_end <= 50].groupby("subject").agg(c=("correct", "mean"), rt=("rt", "median"))
mid = nz[(nz.from_end > 50) & (nz.from_end <= 300)].groupby("subject").agg(c=("correct", "mean"), rt=("rt", "median"))
w1 = stats.wilcoxon(end.c, mid.c)
w2 = stats.wilcoxon(end.rt, mid.rt)
print(f"Percent correct, last 50 trials: median over mice {end.c.median() * 100:.1f}; trials 51 to 300 from the end {mid.c.median() * 100:.1f}; "
      f"lower at the end in {(end.c < mid.c).sum()} of {len(end)} mice, p = {w1.pvalue:.2g}")
print(f"Median first movement time (ms), last 50: {end.rt.median() * 1000:.0f}; trials 51 to 300: {mid.rt.median() * 1000:.0f}; "
      f"longer at the end in {(end.rt > mid.rt).sum()} of {len(end)} mice, p = {w2.pvalue:.2g}")
curve = nz[nz.from_end <= 400].assign(bin=lambda d: (d.from_end - 1) // 50).groupby(["subject", "bin"]).agg(c=("correct", "mean"), rt=("rt", "median"))
print("By 50-trial bins from the end (bin 0 is the last 50), median over mice:")
print(curve.groupby("bin").median().assign(c=lambda d: (d.c * 100).round(1), rt=lambda d: (d.rt * 1000).round(0)).T.to_string())

**Result: reproduced.** In the last 50 trials of the sessions the median over mice of percent correct
on non-zero contrast trials is 80.0%, against 85.0% in trials 51 to 300 from the end (lower in 105 of
139 mice, p = 7.9e-13), and the median first movement time is 279 against 165 ms (longer in 131 of
139 mice, p = 2.6e-22).

## fig1g-block-switch

> After a block switch, mice took around 5 to 10 trials to adjust their behaviour, as shown by the
> fraction of correct choices on 0% contrast trials after the switch.

In [ ]:
t2 = trials.copy()
t2["pos"] = t2.groupby(["eid", "block"]).cumcount() + 1
sw = t2[(t2.block >= 2) & (t2.pos <= 40) & (t2.contrast == 0) & (t2.choice != 0) & t2.eid.isin(BLOCK_OK)]
mouse_curve = sw.groupby(["subject", "pos"]).correct.mean().unstack()
mean_curve = mouse_curve.mean()
n = mean_curve.index.values.astype(float)
f = lambda x, a, b, tau: a - b * np.exp(-(x - 1) / tau)
(a, b, tau), _ = optimize.curve_fit(f, n, mean_curve.values, p0=(0.7, 0.4, 5), maxfev=20000)
print("Mean over mice of the fraction correct on 0% trials, by position after the switch:")
print(mean_curve.round(3).to_string())
print(f"Fit: a = {a:.3f}, b = {b:.3f}, tau = {tau:.2f} trials; 90% of the change at {1 + tau * np.log(10):.1f} trials")
print("Mice with 0% trials at position 1:", int(mouse_curve[1].notna().sum()), "; 0% trials in the curve:", len(sw),
      "; sessions with valid blocks:", len(BLOCK_OK))
fits = []
for k, row in mouse_curve.iterrows():
    try:
        p, _ = optimize.curve_fit(f, n[row.notna()], row.dropna().values, p0=(0.7, 0.4, 5), bounds=([0, -1, 0.1], [1, 1, 100]))
        fits.append(p[2])
    except RuntimeError:
        fits.append(np.nan)
fits = np.array(fits)
print(f"Per-mouse tau: median {np.nanmedian(fits):.2f}, interquartile {np.nanpercentile(fits, 25):.2f} to {np.nanpercentile(fits, 75):.2f}; "
      f"failed fits {int(np.isnan(fits).sum())} of {len(fits)}")
rng_boot = np.random.default_rng(60)
boot = []
mc = mouse_curve.values
for _ in range(1000):
    m = np.nanmean(mc[rng_boot.integers(0, len(mc), len(mc))], 0)
    try:
        boot.append(optimize.curve_fit(f, n, m, p0=(0.7, 0.4, 5), maxfev=20000)[0][2])
    except RuntimeError:
        boot.append(np.nan)
boot = np.array(boot)
lo, hi = np.nanpercentile(boot, [2.5, 97.5])
print(f"Bootstrap over mice (1000 resamples, {int(np.isnan(boot).sum())} failed fits): tau 95% interval {lo:.2f} to {hi:.2f}; "
      f"1 + tau ln 10 interval {1 + lo * np.log(10):.1f} to {1 + hi * np.log(10):.1f} trials; "
      f"resamples with 1 + tau ln 10 in 4.5 to 11: {np.nanmean((1 + boot * np.log(10) >= 4.5) & (1 + boot * np.log(10) <= 11)):.3f}")
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(n, mean_curve.values, "o", ms=3)
ax.plot(n, f(n, a, b, tau))
ax.set_xlabel("trial after block switch (0% contrast)")
ax.set_ylabel("fraction correct")
print("Plotted range:", round(mean_curve.min(), 3), "to", round(mean_curve.max(), 3))

**Result: reproduced.** On 0% contrast trials after a switch between biased blocks (395 sessions
with valid blocks, 16,447 trials), the mean fraction correct starts at 0.338 on the first trial after
the switch. The fitted time constant is 3.86 trials, below the 4.5 to 11 of reading (a); the fit
covers 90% of the change at 9.9 trials, within reading (b). The criterion is met on the point
estimate. Resampling mice gives a 95% interval of 2.79 to 5.30 trials for the time constant and 7.4 to
13.2 trials for reading (b), with 75.9% of resamples within 4.5 to 11. Per-mouse time constants have
a median of 3.88 trials (interquartile 1.86 to 11.30, 125 mice).

## fig1h-block-bias

> For all contrast values mice chose the side of the block's likely stimulus more often, responding
> right more often in right blocks than in left blocks.

In [ ]:
rb = resp[(resp.p_left != 0.5) & resp.eid.isin(BLOCK_OK)]
pr = rb.assign(right=rb.choice > 0).groupby(["subject", "p_left", "signed_contrast"]).right.mean().unstack(["p_left"])
diff = (pr[0.2] - pr[0.8]).unstack("signed_contrast")
rows = []
for c in diff.columns:
    d = diff[c].dropna()
    rows.append({"signed_contrast": c, "mice": len(d), "median_diff": round(d.median(), 3), "higher_in": int((d > 0).sum()),
                 "p": stats.wilcoxon(d).pvalue})
bias = pd.DataFrame(rows)
print(bias.to_string(index=False))
print("Contrasts with a higher right-block fraction (median > 0 and p < 0.05):", int(((bias.median_diff > 0) & (bias.p < 0.05)).sum()), "of 9",
      "; mice:", rb.subject.nunique(), "; sessions with valid blocks:", rb.eid.nunique())
allb = resp[resp.p_left != 0.5].assign(right=lambda d: d.choice > 0).groupby(["subject", "p_left", "signed_contrast"]).right.mean().unstack("p_left")
d_all = (allb[0.2] - allb[0.8]).unstack("signed_contrast")
print("Variant with all sessions, median difference by contrast:", d_all.median().round(3).to_dict())

**Result: reproduced.** In the 395 sessions with valid blocks (125 mice), the fraction of right
choices is higher in right blocks than in left blocks at all nine signed contrasts, with median
differences of 0.013 to 0.333 (largest at 0%) and every p below 1e-8. With all sessions included,
the median differences are smaller at every contrast (0.012 to 0.314).

## Neural data: the regions tested and their spikes

The regions fixed in the claims file: the example regions VISp, GRN and IRN, seven regions drawn at
random from the canonical regions with at most 12 sessions (leaving out the latency regions), and the
latency regions LGd, LP, VISpm and VISam. Each region uses all its canonical sessions and units.

Next, every canonical unit of the 454 canonical sessions is streamed once more, for the trajectories of
all regions (traj-stim-regions) and for the check of the spikes against the trials' clock.

In [ ]:
EXAMPLES = ["VISp", "GRN", "IRN"]
LATENCY = ["LGd", "VISp", "LP", "VISpm", "VISam"]
n_sess = canon.groupby("region").eid.nunique()
pool = sorted(n_sess[n_sess <= 12].index.difference(EXAMPLES + LATENCY))
RANDOM = sorted(np.random.default_rng(409).choice(pool, 7, replace=False).tolist())
DECODING = EXAMPLES + RANDOM
TRAJ = LATENCY + ["IRN", "GRN"]
TESTED = sorted(set(DECODING) | set(TRAJ))
print("Random regions:", RANDOM, "from", len(pool), "candidates")
sel = canon[canon.region.isin(TESTED)]
print(sel.groupby("region").agg(sessions=("eid", "nunique"), units=("unit", "size")).T.to_string())
jobs = []
for aid, g in sel.groupby("asset_id"):
    jobs.append((aid, g.unit.tolist(), g[g.region.isin(TRAJ)].unit.tolist(), g[g.region == "IRN"].unit.tolist()))
t0 = time.time()
extracted = bwm.run_parallel(bwm.extract_session, jobs)
extract_seconds = time.time() - t0
by_asset = {e["asset_id"]: e for e in extracted}
t_eid = dict(zip(processed.asset_id, sessions.eid))
unit_region = {(r.asset_id, r.unit): r.region for r in sel.itertuples()}
print(f"Streamed {len(extracted)} sessions, {sum(e['n_spikes'] for e in extracted) / 1e6:.0f} million spikes of {len(sel)} units, "
      f"in {extract_seconds:.0f} s ({extract_seconds / len(extracted):.2f} s per session with {bwm.N_WORKERS} processes)")
print("Units whose spike times were not sorted:", sum(e["unsorted_units"] for e in extracted))

In [ ]:
# Every canonical unit of every canonical session: trajectory sums for traj-stim-regions (one set of
# permutations per session, shared by its regions, with the Methods' and the code's classes) and the
# clock check.
t0 = time.time()
canon_jobs = [(aid, g.unit.values, g.region.values, 40000 + k) for k, (aid, g) in enumerate(canon.groupby("asset_id", sort=True))]
acc_traj = {k: ({}, {}) for k in ("methods_ok", "methods_all", "code_ok", "code_all")}   # summed squares, cells
clock_all = []
for r in bwm.run_parallel_iter(bwm.region_traj_job, canon_jobs):
    ok = t_eid[r["asset_id"]] in BLOCK_OK
    clock_all.append({"asset_id": r["asset_id"], "n_analysed": r["n_analysed"], "acc": r["clock"]["acc"],
                      "shifted": r["clock"]["shifted"], "block_ok": ok})
    for reg in r["sq"]:
        for key, part in (("methods_ok", "sq"), ("methods_all", "sq"), ("code_ok", "sq_code"), ("code_all", "sq_code")):
            if key.endswith("_ok") and not ok:
                continue
            sq_, n_ = acc_traj[key]
            sq_[reg] = sq_.get(reg, 0) + r[part][reg].astype(np.float64)
            n_[reg] = n_.get(reg, 0) + r["n"][reg]
all_seconds = time.time() - t0
print(f"Streamed and binned {len(canon)} canonical units of {len(canon_jobs)} sessions in {all_seconds:.0f} s; "
      f"sessions with no analysed trial: {sum(c['n_analysed'] == 0 for c in clock_all)}")

In [ ]:
# Spikes against the trials' clock in every canonical session, with all its canonical units.
acc_all = np.array([c["acc"] for c in clock_all])
jump_all = np.nanmean(acc_all[:, bwm.CLOCK_OFFSETS >= 0], 1) - np.nanmean(acc_all[:, bwm.CLOCK_OFFSETS <= -0.4], 1)
print("Balanced accuracy of feedback decoding by window start relative to feedback (s), median over sessions:")
print(dict(zip(bwm.CLOCK_OFFSETS.tolist(), np.nanmedian(acc_all, 0).round(3).tolist())))
print(f"Jump from the windows before to those from feedback on: median {np.nanmedian(jump_all):.3f}; above 0.05 in "
      f"{int(np.sum(jump_all > 0.05))} of {len(jump_all)} sessions; labels shifted by one trial: median "
      f"{np.nanmedian([c['shifted'] for c in clock_all]):.3f}")
bad_all = [processed.set_index("asset_id").loc[c["asset_id"], "path"].split("/")[0] + " " + t_eid[c["asset_id"]][:8]
           for c, j in zip(clock_all, jump_all) if not j > 0.05]
print("Sessions where the jump is not established:", len(bad_all), bad_all)

Feedback becomes decodable in the windows from feedback on (median balanced accuracy 0.90 to 0.94)
and not in the windows before it (0.52 to 0.58), and shifting the labels by one trial removes it
(0.56), so the spike times share the clock of the trials table in the 448 of 454 canonical sessions
where the jump is clear. In the 6 sessions listed above it could not be established with their
canonical units; for them the shared clock is consistent with the data but not shown.

In [ ]:
# Region-session inputs.
def region_counts(region, var_keys=("stim", "choice", "feedback")):
    out = []
    for aid, g in sel[sel.region == region].groupby("asset_id"):
        e = by_asset[aid]
        cols = np.isin(e["units"], g.unit.values)
        out.append((aid, e["trials"], {k: e["counts"][k][:, cols] for k in var_keys}))
    return out

import sklearn
from sklearn.linear_model import LogisticRegression
aid0, t_0, c_0 = region_counts("VISp")[0]
rows0, y0, _ = bwm.decoding_targets(t_0, "stim")
X0 = c_0["stim"][rows0].astype(float)
print("scikit-learn", sklearn.__version__, "; decoder parameters:", {k: v for k, v in bwm._logreg(1.0).get_params().items() if k in ("penalty", "l1_ratio", "solver")})
for C_ in (0.01, 0.1, 1.0):
    l1 = bwm._logreg(C_).fit(X0, y0)
    l2 = LogisticRegression(C=C_, l1_ratio=0.0, solver="liblinear", tol=1e-3, max_iter=20000, class_weight="balanced", random_state=0).fit(X0, y0)
    print(f"   one VISp session, stimulus, C = {C_}: nonzero coefficients {int((l1.coef_ != 0).sum())} of {l1.coef_.size} "
          f"(the same fit with l1_ratio 0, L2: {int((l2.coef_ != 0).sum())})")

t0 = time.time()
dec_jobs = []
for region in DECODING:
    for k, (aid, t, c) in enumerate(region_counts(region)):
        dec_jobs.append(((region, aid), t, c, 1000 * (DECODING.index(region) + 1) + k))
dec = bwm.run_parallel(bwm.decode_job, dec_jobs, workers=4)
decode_seconds = time.time() - t0
rows = []
for d in dec:
    for var in ("stim", "choice", "feedback"):
        rows.append({"region": d["key"][0], "asset_id": d["key"][1], "var": var, **{k: d[var][k] for k in ("n_trials", "n_minor", "n_null", "score", "p", "corrected")}})
dec_df = pd.DataFrame(rows)
print(f"Logistic fits: {sum(d['fits'] for d in dec)}; not converged within 20000 iterations: {sum(d['not_converged'] for d in dec)}")
print(f"{len(dec_jobs)} region-sessions decoded in {decode_seconds:.0f} s; null sessions per decoding: "
      f"{dec_df.n_null.min()} to {dec_df.n_null.max()}; learning rates of the action-kernel model: "
      f"{pd.Series([d['alpha'] for d in dec]).value_counts().sort_index().to_dict()}")
region_res = []
for (region, var), g in dec_df.groupby(["region", "var"]):
    region_res.append({"region": region, "var": var, "sessions": len(g), "p_fisher": bwm.fisher_p(g.p), "effect": g.corrected.median(),
                       "median_score": g.score.median(), "sessions_p05": int((g.p < 0.05).sum())})
region_res = pd.DataFrame(region_res)
region_res["significant"] = False
for var, g in region_res.groupby("var"):
    region_res.loc[g.index, "significant"] = bwm.bh(g.p_fisher.values, 0.01)
ri = region_info.set_index("Beryl")
paper_col = {"stim": "stim_dec", "choice": "choice_dec", "feedback": "fback_dec"}
region_res["paper_effect"] = [ri.loc[r, paper_col[v]] for r, v in zip(region_res.region, region_res["var"])]
region_res["paper_significant"] = [bool(ri.loc[r, paper_col[v] + "_sig"]) for r, v in zip(region_res.region, region_res["var"])]
print(region_res.round(4).sort_values(["var", "region"]).to_string(index=False))
for q in (0.01, 0.05):
    print(f"Variant, BH at q = {q}:", {v: int(bwm.bh(g.p_fisher.values, q).sum()) for v, g in region_res.groupby("var")},
          "; Fisher p < 0.01 uncorrected:", {v: int((g.p_fisher < 0.01).sum()) for v, g in region_res.groupby("var")})
subj_of = dict(zip(processed.asset_id, sessions.subject))
per = dec_df[dec_df.region.isin(EXAMPLES)].assign(subject=lambda d: d.asset_id.map(subj_of))
print("Per session, example regions (score and p), by mouse:")
print(per.pivot_table(index=["region", "subject", "asset_id"], columns="var", values=["score", "p"]).round(3).droplevel(2).to_string())

## dec-stim-restricted

> The visual stimulus side, decoded from the first 100 ms after stimulus onset, was represented in a
> narrower set of regions than choice or feedback, with strong signatures in visual cortex such as
> VISp.

In [ ]:
nsig = region_res.groupby("var").significant.sum()
print("Tested regions significant (BH q = 0.01 over the 10 regions):", nsig.to_dict())
visp = region_res[(region_res.region == "VISp") & (region_res["var"] == "stim")].iloc[0]
print(f"VISp stimulus: {visp.sessions} sessions, Fisher p = {visp.p_fisher:.2g}, effect {visp.effect:.3f} (paper {visp.paper_effect}), "
      f"sessions with p < 0.05: {visp.sessions_p05}")
s = region_res[region_res["var"] == "stim"]
print("Agreement with the paper's significance:", int((s.significant == s.paper_significant).sum()), "of", len(s))
print("Spearman correlation of effects with the paper's:", stats.spearmanr(s.effect, s.paper_effect))
print("Paper over 201 regions: stimulus 40, choice 172, feedback 198 significant")

**Result: reproduced.** Of the 10 tested regions, 4 are significant for stimulus side (CL, GRN, IRN
and VISp), against 5 for choice and 7 for feedback. VISp is significant (35 sessions, Fisher p =
1.3e-08, null-corrected balanced accuracy 0.039, the paper's table 0.02). Each tested region's
significance agrees with the paper's table (10 of 10), and the effects correlate with the paper's
(Spearman 0.91, p = 0.00025). With BH at q = 0.05 the stimulus count stays 4 while choice rises to 7
and feedback to 9. The null that reuses the real labels' C can be narrower than the full nested null
(on three example sessions 95th percentiles of 0.531 to 0.598 against 0.555 to 0.622; VISp stimulus
0.531 against 0.575). For VISp stimulus the nested null was run on all 35 sessions with 30 nulls
each: VISp stays significant (Fisher p 1.8e-05 with the nested null, 2.8e-05 with the reused C on the
same nulls, 13 sessions below 0.05 under each), and remains significant with BH at q = 0.01 among the
10 regions. The decoder fits an L1 penalty: with scikit-learn 1.9.1, at C = 0.01 no coefficient of 13
is nonzero in one VISp session, against 12 with an L2 penalty.

## dec-choice-widespread

> Choice, decoded from the 100 ms before the first wheel movement, was represented in a larger
> number of regions than the visual stimulus, with strong effects in hindbrain regions such as GRN.

In [ ]:
grn = region_res[(region_res.region == "GRN") & (region_res["var"] == "choice")].iloc[0]
print(f"GRN choice: {grn.sessions} sessions, Fisher p = {grn.p_fisher:.2g}, effect {grn.effect:.3f} (paper {grn.paper_effect})")
c = region_res[region_res["var"] == "choice"]
print("Significant for choice:", int(c.significant.sum()), "; for stimulus:", int(nsig["stim"]))
print("Agreement with the paper's significance:", int((c.significant == c.paper_significant).sum()), "of", len(c))
print("Spearman correlation of effects with the paper's:", stats.spearmanr(c.effect, c.paper_effect))
print(c[["region", "sessions", "p_fisher", "effect", "paper_effect", "significant", "paper_significant"]].round(4).to_string(index=False))

**Result: reproduced.** GRN is significant for choice (14 sessions, Fisher p = 6e-15, effect 0.277,
the paper's table 0.28), and 5 tested regions are significant for choice against 4 for stimulus, a
margin of one region. Significance agrees with the paper's table in 9 of 10 regions; LDT (2 sessions,
Fisher p = 0.0154) is significant in the paper and not here. The effects correlate with the paper's
(Spearman 0.98). Here 5 of 10 regions are significant, while the paper reports 172 of 201.

## dec-feedback-nearly-all

> Whether the trial was correct could be decoded from the 200 ms after feedback onset in nearly all
> recorded regions, almost perfectly in many.

In [ ]:
fb = region_res[region_res["var"] == "feedback"]
print("Significant for feedback:", int(fb.significant.sum()), "of", len(fb), "; IRN significant:", bool(fb[fb.region == "IRN"].significant.iloc[0]))
print(fb[["region", "sessions", "p_fisher", "effect", "median_score", "paper_effect", "significant", "paper_significant"]].round(4).to_string(index=False))
f_sess = dec_df[dec_df["var"] == "feedback"]
print(f"Region-sessions with balanced accuracy above 0.9: {int((f_sess.score > 0.9).sum())} of {len(f_sess)}; median accuracy {f_sess.score.median():.3f}")
print("Spearman correlation of effects with the paper's:", stats.spearmanr(fb.effect, fb.paper_effect))

**Result: partially reproduced.** 7 of the 10 tested regions are significant for feedback, including
IRN (27 sessions, effect 0.407, the paper's table 0.42); the criterion asks for 9. HATA (Fisher
p = 0.0255), PA (0.1455) and TR (0.0227) are not significant; the paper's table marks HATA
significant and PA and TR not, so 8 of these 10 regions are significant in the paper. With BH at
q = 0.05, 9 regions are significant. 23 of 111 region-sessions decode feedback with a balanced
accuracy above 0.9, and the median accuracy is 0.913 in GRN and 0.905 in IRN. The effects correlate
with the paper's (Spearman 0.96).

In [ ]:
# Variant for the three decoding claims: the null with the full nested search of C, against the
# null that reuses the real labels' C, on three VISp, GRN and IRN sessions with 20 nulls each.
nn_jobs = []
for k, region in enumerate(EXAMPLES):
    aid, t, c = region_counts(region)[0]
    nn_jobs.append(((region, aid), t, c, 5000 + k, 20))
nn = bwm.run_parallel(bwm.nested_null_job, nn_jobs, workers=3)
for r in nn:
    for var in ("stim", "choice", "feedback"):
        a, b = r[var]["nested"], r[var]["reused"]
        print(f"{r['key'][0]:5s} {var:8s} nested null: median {np.median(a):.3f}, 95th pct {np.percentile(a, 95):.3f}; "
              f"reused C: median {np.median(b):.3f}, 95th pct {np.percentile(b, 95):.3f}")

In [ ]:
# Variant for dec-stim-restricted: VISp stimulus with the full nested search for every null, on the
# first 30 of the same pseudo-sessions in each of its sessions, against the reused C on those 30.
N_NESTED = 30
visp_jobs, reused_p = [], []
for d, job in zip(dec, dec_jobs):
    if d["key"][0] != "VISp":
        continue
    visp_jobs.append((d["key"], job[1], job[2], job[3], d["stim"]["score"], N_NESTED))
    nl = d["stim"]["null"][:N_NESTED]
    reused_p.append((1 + np.sum(nl >= d["stim"]["score"])) / (1 + len(nl)))
t0 = time.time()
vn = bwm.run_parallel(bwm.nested_p_job, visp_jobs, workers=4)
nested_p = np.array([r["p"] for r in vn])
reused_p = np.array(reused_p)
print(f"VISp stimulus, {len(vn)} sessions, {N_NESTED} nulls each, in {time.time() - t0:.0f} s")
print(f"   per-session p, nested null: median {np.median(nested_p):.3f}, p < 0.05 in {int((nested_p < 0.05).sum())}; "
      f"reused C: median {np.median(reused_p):.3f}, p < 0.05 in {int((reused_p < 0.05).sum())}")
print(f"   95th percentile of the null, median over sessions: nested {np.median([np.percentile(r['null'], 95) for r in vn]):.3f}, "
      f"reused {np.median([np.percentile(d['stim']['null'][:N_NESTED], 95) for d in dec if d['key'][0] == 'VISp']):.3f}")
fp_nested, fp_reused = bwm.fisher_p(nested_p), bwm.fisher_p(reused_p)
s_p = region_res[region_res["var"] == "stim"].set_index("region").p_fisher.copy()
s_p["VISp"] = fp_nested
print(f"   Fisher p, nested null {fp_nested:.2g}, reused C on the same {N_NESTED} nulls {fp_reused:.2g}; "
      f"VISp significant with BH at q = 0.01 among the 10 stimulus p-values when its p is the nested one: "
      f"{bool(bwm.bh(s_p.values, 0.01)[list(s_p.index).index('VISp')])}")
print(pd.DataFrame({"session": [r["key"][1][:8] for r in vn], "score": [r_["stim"]["score"] for r_ in dec if r_["key"][0] == "VISp"],
                    "p_reused": reused_p, "p_nested": nested_p}).round(3).to_string(index=False))

## Single-cell selectivity (sc-stim-fraction, sc-choice-fraction)

The code's test on the units of all 14 loaded regions, with a null run on one pseudo-session (stimulus)
or synthetic session (choice) per region-session to measure the false-positive rate.

In [ ]:
t0 = time.time()
sc_jobs = []
for region in TESTED:
    for k, (aid, t, c) in enumerate(region_counts(region, ("stim", "choice"))):
        if t_eid[aid] not in BLOCK_OK:
            continue
        sc_jobs.append(((region, aid), t, c, 20000 + 100 * TESTED.index(region) + k))
sc = bwm.run_parallel(bwm.single_cell_job, sc_jobs, workers=4)
left_out = sel[~sel.eid.isin(BLOCK_OK)]
print(f"Left out for invalid blocks: {left_out.groupby(['region', 'eid']).ngroups} region-sessions, {len(left_out)} units")
rows = []
for r in sc:
    for i in range(len(r["stim_real"])):
        rows.append({"region": r["key"][0], "asset_id": r["key"][1], **{k: bool(r[k][i]) for k in r if k != "key"}})
cells = pd.DataFrame(rows)
print(f"{len(cells)} units in {len(sc_jobs)} region-sessions, in {time.time() - t0:.0f} s")
frac = cells.groupby("region")[["stim_real", "choice_real", "stim_null", "choice_null", "stim_shuf", "choice_shuf"]].mean()
frac["units"] = cells.groupby("region").size()
frac["paper_stim"] = [ri.loc[r, "stim_man"] for r in frac.index]
frac["paper_choice"] = [ri.loc[r, "choice_man"] for r in frac.index]
print(frac.round(4).to_string())

## sc-stim-fraction

> Single-cell tests controlling for choice and block found few stimulus-selective neurons (0.5% of
> all neurons); even in the regions with the largest fractions, such as visual cortex, the fraction
> did not exceed about 10%.

In [ ]:
def ci(k, n):
    lo = stats.beta.ppf(0.025, k, n - k + 1) if k > 0 else 0.0
    hi = stats.beta.ppf(0.975, k + 1, n - k)
    return lo, hi

for var in ("stim", "choice"):
    k, n = int(cells[f"{var}_null"].sum()), len(cells)
    lo, hi = ci(k, n)
    print(f"{var}: null run selects {k} of {n} units ({k / n:.2e}), 95% interval {lo:.2e} to {hi:.2e}; "
          f"paper's rate 5e-5, band 1.7e-5 to 1.5e-4; interval excludes the band: {lo > 1.5e-4 or hi < 1.7e-5}")
    print(f"   null units passing the simple test alone: {cells[f'{var}_null_mw'].mean():.2e}; the combined test alone: {cells[f'{var}_null_cc'].mean():.3f}")
    k2 = int(cells[f"{var}_shuf"].sum())
    lo2, hi2 = ci(k2, n)
    print(f"   labels shuffled over the tested trials (one permutation per region-session): {k2} of {n} units ({k2 / n:.2e}), "
          f"95% interval {lo2:.2e} to {hi2:.2e}; simple test alone {cells[f'{var}_shuf_mw'].mean():.2e}, combined test alone "
          f"{cells[f'{var}_shuf_cc'].mean():.3f}; product of the two single rates {cells[f'{var}_shuf_mw'].mean() * cells[f'{var}_shuf_cc'].mean():.1e}")
print("\nStimulus: largest regional fraction", round(frac.stim_real.max(), 4), "in", frac.stim_real.idxmax(),
      "; pooled over the 14 regions", round(cells.stim_real.mean(), 4))
agree = (frac.stim_real - frac.paper_stim).abs() <= 0.02
print("Regions within 0.02 of the paper's fraction:", int(agree.sum()), "of", len(frac))
print("Paper, pooled over the canonical regions (from region_info.csv):",
      round(float((region_info[region_info.canonical].stim_man * region_info[region_info.canonical]["# good neurons"]).sum()
                  / region_info[region_info.canonical]["# good neurons"].sum()), 4))

**Result: blocked (methods underspecified).** The test ran, but the null run selects 14 of 5,228 units
(2.7e-3, 95% interval 1.5e-3 to 4.5e-3), far above the paper's false-positive rate of 5e-5, so by the
criterion the procedures differ. Under the null the simple test alone passes 7.1e-3 of units and the
combined test 6.9%. With the stimulus side shuffled over the tested trials, the joint test selects 4
units (7.7e-4, interval 2.1e-4 to 2.0e-3), while the product of the two tests' single rates is 4.7e-5,
close to the paper's 5e-5. The two tests are not independent, as the paper's rate assumes, and that
alone puts the rate above the band; the pseudo-session null is about 3.5 times higher again. The
fractions themselves agree with the paper's table: all 14 regions are within 0.02 of it, the largest
is 5.9% (CL, 17 units), and the visual regions LGd, VISp, VISpm and VISam lie between 2.1% and 3.8%.
Pooled over the 14 regions the fraction is 1.82%, against 0.41% in the paper's table over all
canonical regions; the tested regions over-represent visual areas.

## sc-choice-fraction

> More neurons responded to choice than to the visual stimulus (4% of all neurons), with the largest
> fractions in hindbrain, midbrain and thalamic regions.

In [ ]:
only_c = int((cells.choice_real & ~cells.stim_real).sum())
only_s = int((cells.stim_real & ~cells.choice_real).sum())
bt = stats.binomtest(only_c, only_c + only_s, 0.5, alternative="greater") if only_c + only_s else None
print(f"Pooled: choice {cells.choice_real.mean():.4f}, stimulus {cells.stim_real.mean():.4f}; selective for choice only {only_c}, "
      f"stimulus only {only_s}, binomial p = {bt.pvalue if bt else float('nan'):.2g}")
print("Spearman correlation of regional choice fractions with the paper's:", stats.spearmanr(frac.choice_real, frac.paper_choice))
print(frac[["units", "choice_real", "paper_choice"]].sort_values("choice_real", ascending=False).round(4).to_string())

**Result: blocked (methods underspecified).** The null run with synthetic choices selects 13 of 5,228
units (2.5e-3, 95% interval 1.3e-3 to 4.2e-3), far above 5e-5; with the choices shuffled over the
tested trials it selects 3 (5.7e-4, interval 1.2e-4 to 1.7e-3), against a product of single rates of
3.3e-5. Apart from that, the claimed pattern is present: pooled over the 14 regions 5.64% of units are
choice-selective against 1.82% for stimulus (270 units selective for choice only, 70 for stimulus
only, p = 3.8e-29), the largest fractions are in CL (35%), GRN (22%) and IRN (14%), and the regional
fractions correlate with the paper's (Spearman 0.84, p = 0.00016).

## Population trajectories (traj-latency-visual, traj-latency-late)

In [ ]:
t0 = time.time()
traj_res, traj_all, traj_long = {}, {}, {}
for k, region in enumerate(TRAJ):
    sess, sess_all = [], []
    for aid, g in sel[sel.region == region].groupby("asset_id"):
        e = by_asset[aid]
        cols = np.isin(e["traj_units"], g.unit.values)
        t = e["trials"]
        a = t.analysed.values
        item = (e["traj"][a][:, cols], t.side.values[a], t.p_left.values[a], t.choice.values[a])
        sess_all.append(item)
        if t_eid[aid] in BLOCK_OK:
            sess.append(item)
    short = [(b[:, :, :bwm.TRAJ_TIMES.size],) + tuple(rest) for b, *rest in sess]
    traj_res[region] = bwm.trajectory(short, bwm.N_PERM_TRAJ, np.random.default_rng(3000 + k))
    traj_all[region] = bwm.trajectory([(b[:, :, :bwm.TRAJ_TIMES.size],) + tuple(rest) for b, *rest in sess_all], bwm.N_PERM_TRAJ,
                                      np.random.default_rng(3100 + k))
    traj_long[region] = bwm.trajectory(sess, bwm.N_PERM_TRAJ, np.random.default_rng(3200 + k), bwm.TRAJ_TIMES_LONG)
def as_table(res):
    df = pd.DataFrame({r: {k: v for k, v in d.items() if k != "d"} for r, d in res.items()}).T
    for c in ("latency", "latency_70", "latency_70_bin_start", "latency_canonical"):
        df[c] = (df[c].astype(float) * 1000).round(1)
    return df
traj_df = as_table(traj_res)
print("Sessions left out for invalid blocks:", {r: int(sel[(sel.region == r) & ~sel.eid.isin(BLOCK_OK)].eid.nunique()) for r in TRAJ})
print(f"Trajectories of {len(TRAJ)} regions with {bwm.N_PERM_TRAJ} permutations, three variants, in {time.time() - t0:.0f} s")
print("latency_70: first time above 70% of the range (Methods); latency: also significant point-wise (code); ms on the code's axis;")
print("latency_canonical: the code's canonical 0 to 100 ms window (first 48 points, full-window baseline) with significance")
print(traj_df.to_string())
print("\nVariant with the 64 sessions of invalid blocks included (permutations within their recorded blocks):")
print(as_table(traj_all)[["n_cells", "amp", "p", "latency_70", "latency"]].to_string())
print("\nVariant with the window 0 to 200 ms (96 points, axis linspace(0, 0.2, 96)):")
print(as_table(traj_long)[["n_cells", "amp", "p", "latency_70", "latency"]].to_string())
print("\nDistance (spikes/s, minimum subtracted) every 10th point of the 0 to 200 ms window, by time (ms):")
print(pd.DataFrame({r: d["d"][::10] for r, d in traj_long.items()}, index=(bwm.TRAJ_TIMES_LONG[::10] * 1000).round(1)).round(2).T.to_string())
fig, ax = plt.subplots(figsize=(6, 3.5))
for r, d in traj_res.items():
    ax.plot(bwm.TRAJ_TIMES * 1000, d["d"], label=r)
ax.set_xlabel("time from stimulus onset (ms)")
ax.set_ylabel("distance (spikes/s)")
ax.legend(fontsize=7)
print("Plotted distances range from 0 to", round(max(d["d"].max() for d in traj_res.values()), 2))

## traj-latency-visual

> The latency of the stimulus trajectory separation ordered the visual regions as LGd < VISp, which is
> about equal to LP, < VISpm < VISam, at about 34, 42, 42, 57 and 78 ms.

In [ ]:
exp = {"LGd": 34, "VISp": 42, "LP": 42, "VISpm": 57, "VISam": 78}
for col in ("latency_70", "latency", "latency_canonical"):
    L = traj_df[col].astype(float)
    order = (L["LGd"] < min(L["VISp"], L["LP"]) and max(L["VISp"], L["LP"]) < L["VISpm"] < L["VISam"]
             and abs(L["VISp"] - L["LP"]) <= 10)
    within = {r: bool(abs(L[r] - v) <= 0.1 * v) for r, v in exp.items()}
    print(f"{col:11s}", {r: L[r] for r in exp}, "order holds:", order, "within 10%:", within)
L = as_table(traj_all)["latency_70"].astype(float)
print("Variant with the 64 sessions included, latency_70:", {r: L[r] for r in exp}, "order holds:",
      bool(L["LGd"] < min(L["VISp"], L["LP"]) and max(L["VISp"], L["LP"]) < L["VISpm"] < L["VISam"] and abs(L["VISp"] - L["LP"]) <= 10),
      "within 10%:", {r: bool(abs(L[r] - v) <= 0.1 * v) for r, v in exp.items()})
print("Significance of the distance (permutation p):", traj_df.loc[list(exp), "p"].to_dict())
paper_tab = pd.read_pickle(io.BytesIO(requests.get(f"{REPO}/brainwidemap/meta/Results/stim.pkl", timeout=60).content))
print("The paper's results table meta/Results/stim.pkl, euclidean_latency (ms):",
      (paper_tab.set_index("region").loc[list(exp) + ["IRN", "GRN"], "euclidean_latency"] * 1000).round(1).to_dict())

**Result: not reproduced.** The latencies (Methods' definition) are LGd 31.7, VISp 46.5, LP 48.6,
VISpm 61.3 and VISam 59.2 ms. LGd comes first and VISp and LP are within 10 ms of each other, but
VISam is not later than VISpm, and VISp, LP and VISam are outside 10% of the paper's 42, 42 and 78 ms.
The code's significance condition and its 0 to 100 ms window (with the full window's baseline, as in
the code) change the values by less than 1 ms. Including the 16 region-sessions with invalid blocks
leaves all five latencies as they are. The paper's own results table gives 34.0, 44.7, 46.8, 59.6
and 66.0 ms; for VISp, LP, VISpm and VISam it is closer to the values here than the text is. All five
distances are significant (p = 0.001).

## traj-latency-late

> The early visual wave was followed by later stimulus encoding in the MRN, SCm, PRNr, IRN and GRN,
> with latencies of about 100 to 120 ms.

In [ ]:
for r in ("IRN", "GRN"):
    d = traj_df.loc[r]
    print(f"{r}: latency (Methods) {d.latency_70} ms, with significance (code) {d.latency} ms, distance p = {d.p:.3g}, "
          f"{int(d.n_cells)} cells; in 90 to 132 ms: {90 <= d.latency_70 <= 132}")
long_df = as_table(traj_long)
for r in ("IRN", "GRN"):
    print(f"{r}, variant 0 to 200 ms: latency (Methods) {long_df.loc[r, 'latency_70']} ms, with significance {long_df.loc[r, 'latency']} ms; "
          f"canonical 0 to 100 ms window with significance {traj_df.loc[r, 'latency_canonical']} ms")

**Result: reproduced.** The IRN distance reaches 70% of its range at 118.3 ms and the GRN distance at
109.9 ms, both within 90 to 132 ms, and both distances are significant (p = 0.001). With the window
extended to 200 ms the latencies are 117.9 and 109.5 ms, so they do not depend on where the window
ends here, though the IRN distance is still rising at 190 ms. In the code's canonical 0 to 100 ms
window IRN has no latency and GRN has 100.0 ms, and the paper's results table, which uses that window,
gives 89.4 ms for both. MRN, SCm and PRNr were not tested.

## traj-stim-regions

> The stimulus trajectory distance was significant in 104 regions.

The summed squares of every canonical region were computed in the pass over all canonical sessions
above.

In [ ]:
def region_table(sq, ncell):
    rows = []
    for reg in canon_regions:
        n_c = ncell.get(reg, 0)
        st = bwm.traj_stats(sq[reg], n_c) if n_c >= 20 else {}
        rows.append({"region": reg, "n_cells": n_c, "amp": st.get("amp", np.nan), "p": st.get("p", np.nan),
                     "latency_70": st.get("latency_70", np.nan) * 1000})
    df = pd.DataFrame(rows).set_index("region")
    tested = df.p.notna()
    for name, fn, q in (("bh_01", bwm.bh, 0.01), ("bh_05", bwm.bh, 0.05), ("by_05", bwm.by, 0.05)):
        df[name] = False
        df.loc[tested, name] = fn(df.p[tested].values, q)
    df["raw_01"] = df.p < 0.01
    df["raw_05"] = df.p < 0.05
    df["paper"] = [bool(ri.loc[r, "stim_euc_sig"]) for r in df.index]
    return df

tables = {k: region_table(*v) for k, v in acc_traj.items()}
traj_main = tables["methods_ok"]
tested = traj_main.p.notna()
print(f"Canonical regions {len(traj_main)}; tested (20 or more units in sessions with valid blocks) {int(tested.sum())}; "
      f"not tested {sorted(traj_main.index[~tested])}")
print(f"Smallest attainable p: {1 / (bwm.N_PERM_TRAJ + 1):.4f}; regions at it: {int((traj_main.p <= 1 / (bwm.N_PERM_TRAJ + 1) + 1e-12).sum())}")
print(f"Paper: 104 (region_info.csv stim_euc_sig marks {int(traj_main.paper.sum())} of these {len(traj_main)} regions)")
names = {"methods_ok": "Methods' classes, valid-block sessions (main)", "methods_all": "Methods' classes, all 454 sessions",
         "code_ok": "code's classes, valid-block sessions", "code_all": "code's classes, all 454 sessions"}
labels = {"bh_01": "(a) BH q = 0.01", "raw_01": "(b) P < 0.01", "by_05": "code, BY at 0.05", "bh_05": "BH q = 0.05", "raw_05": "P < 0.05"}
rows = []
for k, df in tables.items():
    for c, lab in labels.items():
        s_ = df[c]
        rows.append({"permutations and sessions": names[k], "threshold": lab, "significant": int(s_.sum()),
                     "within 104 +/- 10.4": abs(int(s_.sum()) - 104) <= 10.4, "agree with paper": int((s_ == df.paper).sum()),
                     "both": int((s_ & df.paper).sum()), "here only": int((s_ & ~df.paper).sum()), "paper only": int((~s_ & df.paper).sum())})
print(pd.DataFrame(rows).to_string(index=False))
chk = pd.DataFrame({"amp_all_regions_pass": traj_main.loc[TRAJ, "amp"], "amp_trajectory_section": traj_df.loc[TRAJ, "amp"].astype(float)})
print("Amplitude of the seven trajectory regions in this pass and in the section above (same cells and trials):")
print(chk.round(4).T.to_string())
print("Median p over the tested regions: Methods' classes", round(float(traj_main.p.median()), 4), "; code's classes, all sessions",
      round(float(tables["code_all"].p.median()), 4))
print("Regions significant in the paper's column and not under (a), with their p here:",
      traj_main[traj_main.paper & ~traj_main.bh_01].p.round(4).to_dict())

**Result: not reproduced.** Of the 201 canonical regions, 195 have 20 or more units in sessions with
valid blocks. Their stimulus trajectory distance is significant in 46 regions with
Benjamini-Hochberg at q = 0.01 (reading a) and in 62 at P < 0.01 (reading b), against the paper's
104 +/- 10.4; 42 regions reach the smallest attainable p of 0.001. Including the 64 sessions with
invalid blocks gives 51 and 70. Permuting within the code's classes, which merge the unbiased block
with the 0.2 blocks, gives 47 and 67, and 53 and 76 with all sessions. The code's Benjamini-Yekutieli
correction at 0.05 gives the same counts as reading (a). Only an uncorrected P < 0.05 comes near the
paper (91 to 98 regions). Significance agrees with the paper's `stim_euc_sig` column in 133 of 201
regions: 41 are significant in both, 5 here only and 63 in the paper only. The seven regions of the
section above have the same amplitudes in this pass.

## fback-oscillation-licking

> During reward delivery, activity showed coherent oscillations of about 10 Hz that were missing after
> negative feedback and were closely related to licking.

In [ ]:
rows = []
lick_by_asset = {s["asset_id"]: np.sort(s["licks"]) for s in scans}
for aid, g in sel[sel.region == "IRN"].groupby("asset_id"):
    e = by_asset[aid]
    t = e["trials"]
    pop = e["psth"].mean(0)          # 2 x bins: correct, incorrect
    f_c, p_c, freqs = bwm.spectrum_peak(pop[0], bwm.PSTH_BIN)
    _, p_i, _ = bwm.spectrum_peak(pop[1], bwm.PSTH_BIN)
    band = np.abs(freqs - f_c) <= 1
    in_band = (freqs >= 4) & (freqs <= 20)
    # Variants: each unit's PSTH z-scored over both conditions; power as a fraction of the 4 to 20 Hz power
    z = e["psth"]
    z = (z - z.mean((1, 2), keepdims=True)) / np.where(z.std((1, 2), keepdims=True) > 0, z.std((1, 2), keepdims=True), 1)
    zf_c, zp_c, _ = bwm.spectrum_peak(z.mean(0)[0], bwm.PSTH_BIN)
    _, zp_i, _ = bwm.spectrum_peak(z.mean(0)[1], bwm.PSTH_BIN)
    zband = np.abs(freqs - zf_c) <= 1
    lk = lick_by_asset[aid]
    fb = t.feedback_time[t.analysed & t.correct].values
    edges = fb[:, None] + np.arange(0, bwm.PSTH_POST, bwm.PSTH_BIN)[None, :]
    lick_psth = bwm.window_counts(lk, edges, bwm.PSTH_BIN).mean(0) / bwm.PSTH_BIN if len(lk) else np.full(edges.shape[1], np.nan)
    f_l = bwm.spectrum_peak(lick_psth, bwm.PSTH_BIN)[0] if len(lk) else np.nan
    rows.append({"session": aid[:8], "units": len(g), "peak_correct_hz": f_c, "power_correct": p_c[band].sum(),
                 "power_incorrect": p_i[band].sum(), "licks": len(lk), "lick_peak_hz": f_l,
                 "error_trials": int((t.analysed & ~t.correct).sum()),
                 "rel_correct": p_c[band].sum() / p_c[in_band].sum(), "rel_incorrect": p_i[band].sum() / p_i[in_band].sum(),
                 "z_peak_hz": zf_c, "z_power_correct": zp_c[zband].sum(), "z_power_incorrect": zp_i[zband].sum()})
osc = pd.DataFrame(rows)
print(osc.round(2).to_string(index=False))
w = stats.wilcoxon(osc.power_correct, osc.power_incorrect)
print(f"Median peak frequency, correct trials: {osc.peak_correct_hz.median():.2f} Hz; power at the peak higher on correct trials in "
      f"{int((osc.power_correct > osc.power_incorrect).sum())} of {len(osc)} sessions, Wilcoxon p = {w.pvalue:.2g}")
print(f"Sessions whose correct-trial peak is at the 4 Hz edge of the band (below 4.1 Hz): {int((osc.peak_correct_hz < 4.1).sum())} of {len(osc)}; "
      f"peak frequencies range {osc.peak_correct_hz.min():.2f} to {osc.peak_correct_hz.max():.2f} Hz")
for name, a_, b_ in (("power as a fraction of the 4 to 20 Hz power", "rel_correct", "rel_incorrect"),
                     ("PSTHs z-scored per unit", "z_power_correct", "z_power_incorrect")):
    w_ = stats.wilcoxon(osc[a_], osc[b_])
    print(f"Variant, {name}: higher on correct trials in {int((osc[a_] > osc[b_]).sum())} of {len(osc)} sessions, Wilcoxon p = {w_.pvalue:.2g}; "
          f"median ratio correct / incorrect {np.median(osc[a_] / osc[b_]):.2f}")
print(f"Variant, PSTHs z-scored per unit: median peak frequency {osc.z_peak_hz.median():.2f} Hz")
print(f"Median lick PSTH peak: {osc.lick_peak_hz.median():.2f} Hz over {int(osc.lick_peak_hz.notna().sum())} sessions with licks")

**Result: partially reproduced.** In 27 IRN sessions, the population PSTH after correct feedback has
a median peak frequency of 7.08 Hz, below the 9 to 11 Hz of the criterion. Power at each session's
peak is higher after correct than after incorrect feedback in 20 of 27 sessions (p = 0.0042), and the
lick PSTH peaks at a median 7.23 Hz, within 1 Hz of the neural peak. In 4 sessions the neural peak
lies at the 4 Hz edge of the band searched (peaks range 4.00 to 15.62 Hz). The power difference does
not rest on the larger evoked response after reward alone: with each unit's PSTH z-scored it holds in
18 of 27 sessions (p = 0.028, median peak 7.28 Hz), and power at the peak as a fraction of the 4 to
20 Hz power is higher after correct feedback in 26 of 27 sessions (p = 1e-07).

## Claims over all regions (blocked, not attempted)

wheel-speed-decoding, encoding-feedback-largest, video-correlates and granger-interactions need
every canonical region, and models fitted far more often than the trajectories above. They were not
attempted in this notebook; the cost measured here sets the scale for a later notebook.

In [ ]:
per_session = extract_seconds / len(extracted)
per_decode = decode_seconds * 4 / len(dec_jobs) / 3            # CPU seconds per variable per region-session
n_region_sessions = canon.groupby(["region", "eid"]).ngroups
print(f"Streaming the spikes of all {canon.eid.nunique()} canonical sessions at {per_session:.2f} s each: about {canon.eid.nunique() * per_session / 60:.0f} min")
print(f"Decoding one variable for all {n_region_sessions} region-sessions as here (1 run, 100 nulls with reused C): about "
      f"{n_region_sessions * per_decode / 4 / 60:.0f} min on 4 processes; with the paper's 10 runs and 200 nested-search nulls the "
      f"null fits alone grow by more than 10 x 2 x 35 times")
print(f"Streaming all {len(canon)} canonical units of {len(canon_jobs)} sessions with their trajectories and clock check took {all_seconds / 60:.1f} min")
print(f"Total run time of this notebook so far: {(time.time() - started) / 60:.1f} min of the 30 min limit")

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and
the differences from the paper's method.

| Claim | Outcome |
|---|---|
| dataset-sessions-mice-labs | reproduced |
| dataset-probes-units | reproduced |
| well-isolated-neurons | reproduced |
| canonical-regions | reproduced |
| brain-areas-279 | reproduced |
| mice-sex | reproduced |
| mice-age-weight | partially reproduced (weight yes, age no) |
| probe-versions | reproduced |
| trials-per-session | reproduced |
| rf-mapping | blocked, data missing |
| behav-performance | reproduced (mean over sessions) |
| behav-zero-contrast | partially reproduced |
| behav-early-movements | reproduced |
| fig1d-contrast | reproduced |
| fig1ef-end-of-session | reproduced |
| fig1g-block-switch | reproduced (reading b) |
| fig1h-block-bias | reproduced |
| dec-stim-restricted | reproduced |
| dec-choice-widespread | reproduced |
| dec-feedback-nearly-all | partially reproduced |
| sc-stim-fraction | blocked, methods underspecified |
| sc-choice-fraction | blocked, methods underspecified |
| traj-latency-visual | not reproduced |
| traj-latency-late | reproduced |
| traj-stim-regions | not reproduced (46 regions) |
| fback-oscillation-licking | partially reproduced |
| wheel-speed-decoding | blocked, not attempted |
| encoding-feedback-largest | blocked, not attempted |
| video-correlates | blocked, not attempted |
| granger-interactions | blocked, not attempted |

**How the analysis differs from the paper's.**

- The decoding, single-cell and latency claims are tested on 14 regions fixed in advance (10 for
  decoding), each with all its canonical sessions, not on all 201 regions; the significance correction
  runs over the tested regions. The count of regions with stimulus trajectories uses all 201.
- Decoding uses one run of the nested cross-validation (paper 10) and 100 null sessions (paper 200),
  and each null reuses the C chosen for the real labels; on three example sessions this null is
  narrower than the full nested null in 6 of 9 comparisons of the 95th percentiles. For VISp stimulus, over all its sessions,
  the conclusion is the same under both nulls.
- The synthetic sessions for choice and feedback come from an action-kernel model fitted here, with
  a logistic choice policy, in place of the companion paper's model package.
- The paper's trial mask uses saturation intervals and data revisions that are not in the dandiset;
  the mask here has the Methods' criteria and the code's truncation.
- 64 sessions whose block column does not follow the stimuli are left out of the block analyses.
- Trajectory permutations follow the Methods' classes (each block apart); the code merges the
  unbiased block with the 0.2 blocks, shown as a variant.

**What was found about the dandiset.**

- It holds the released dataset completely: 459 sessions, 139 mice, 12 labs, 699 insertions,
  621,733 units, 75,708 well-isolated units, and every count of the paper's inclusion table.
- In 64 sessions (protocol name `_iblrig_tasks_ephysChoiceWorld` without a version)
  `probability_left`, and so `block_index` and `block_type`, alternate on every trial after the first
  90 and do not describe the blocks: the stimulus is on the stated likely side in 47.5% to 57% of
  trials, against 76% to 83% elsewhere.
- No session holds the receptive-field mapping phase or its stimulus.
- The quality score matches a noise cutoff below 5 for every unit; the Methods give 20 uV.
- The first movement time is described as after the go cue but precedes it in 12% of trials; the
  reward volume is 1.5 uL on every rewarded trial, where the Methods give 3 uL.
- Subject weight is one value per mouse, and the ages at recording (13.1 to 65.9 weeks) do not reach
  the paper's 178 weeks.
- The spikes share the trials' clock in 448 of 454 canonical sessions; in the other 6 it could not be
  shown.